# ICU Mortality Prediction

**Hands-on Machine Learning — Final project**

**Group members:** Fatoumata OULARE, Lydia BENOSMANE, Marina GELLER YAMAGUTI

**Data:** [WiDS Datathon 2020 (Kaggle)](https://www.kaggle.com/competitions/widsdatathon2020/data)

**How to run**

*   Download `training_v2.csv`, `unlabeled.csv` and `WiDS Datathon 2020 Dictionary.csv` into `data/raw/`

*   Python 3.12 with the packages of `requirements.txt`

*   A full run takes about 20 minutes on a laptop

---

## 1. Introduction

### 1.1 Task

When a patient arrives in an intensive care unit (ICU), the team records vital signs, lab results, the reason for admission and a few chronic conditions. Our task is to estimate the probability that the patient will **die before leaving the hospital**, using **only what is known after the first 24 hours of the ICU stay**.
The target is only known at discharge while the features are routine measurements. That is the setting where a predictive model is useful, for example to flag high-risk patients early. It is a **binary classification** problem on **tabular data**. Hospitals already use a severity score, **APACHE IVa**, a fixed formula computed from the same first-day data. A useful model has to be at least as good, and to **work in hospitals it has never seen**

### 1.2 Metrics

Only about **9%** of the patients die, so the **accuracy** is misleading. Predicting "survives" for everybody already gives about **91%**.

We use instead:

*   **ROC-AU** : probability that a random death is ranked above a random survivor (0.5 = mean, 1 = perfect). It's the official Kaggle metric.

*   **PR-AUC** (average precision): focuses on the rare class, chance level = death rate (~0.09)

*   **Recall** and **precision** at a chosen threshold:

*   **Brier score** and **calibration curve** : are the predicted probabilities numerically correct?

### 1.3 State of the art

The competition (January–February 2020) attracted 951 teams. Scores are ROC-AUC on Kaggle's private test set, whose labels were never released.

| Source | ROC-AUC | Validation | Main methods |
|:--|--:|:--|:--|
| [1st — Women Power](https://www.kaggle.com/competitions/widsdatathon2020/writeups/women-power-1st-place-approach) | 0.91497 | not grouped | Large ensemble (CatBoost, XGBoost, LightGBM, neural nets), 3-level stacking, pseudo-labelling |
| [2nd — nullset](https://www.kaggle.com/competitions/widsdatathon2020/writeups/nullset-2nd-place-solution) | 0.91485 | 8-fold | 26 submissions blended, group statistics by hospital |
| [3rd — Prevision.io](https://www.kaggle.com/competitions/widsdatathon2020/writeups/prevision-io-3rd-place-solution) | 0.91445 | repeated stratified K-fold | 2 LightGBM + 1 neural net, pseudo-labelling |
| [6th — Dihydrogen Oxide](https://www.kaggle.com/competitions/widsdatathon2020/writeups/dihydrogen-oxide-6th-place-solution) | 0.91395 | **grouped by hospital** | Stacking of XGBoost and LightGBM |
| [14th — u++ & mtmt](https://www.kaggle.com/competitions/widsdatathon2020/writeups/u-mtmt-14th-place-approach-summary) | 0.91271 (best single LightGBM: 0.9097 in CV) | not stated | Stacking of 55–73 models |
| [Ren, Zhao & Zhang (2022)](https://doi.org/10.1155/2022/3938492) | 0.879 | not available | Stacked ensemble |
| [Caires Silveira et al. (2022)](https://doi.org/10.5492/wjccm.v11.i5.317) | 0.85 | 80/20 random split, 1,087 patients | Random forest |

We see that:

*   The leaderboard is very tight: the 50th team scored 0.91013, only 0.005 below the winner. The gains at the top come from large ensembles and test-set tricks

*   Most top teams **dropped the APACHE predictions and the hospital identifiers**

*   None of them reports calibration, a decision threshold, or performance on new hospitals

*   Our scores are **not directly comparable** with the leaderboard: the Kaggle test labels are not public, so we evaluate on our own held-out hospitals

### 1.4 Our approach

We do not try to win the leaderboard. Following our teacher's feedback, we answer three questions:

1.  **Does the model overfit the training hospitals?** We evaluate on **entire unseen hospitals** and compare with the performance on the training patients

2.  **What other sources of overfitting are there?** Model complexity, hospital "fingerprints" in the features, and preprocessing fitted on all the data

3.  **How close to the state of the art can simple models get with a careful preprocessing?**

APACHE IVa is kept as a **clinical benchmark**, not as a feature.

---

## 2. Preprocessing

In [ ]:
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn import (
    base,
    calibration,
    compose,
    dummy,
    ensemble,
    impute,
    linear_model,
    metrics,
    model_selection,
    pipeline,
    preprocessing,
)

In [ ]:
%config InlineBackend.figure_format="retina"  # For high DPI display

sns.set_style("darkgrid")
sns.set_context("notebook")
warnings.filterwarnings("ignore")  # Hide library warnings

RANDOM_STATE = 42

print(sklearn.__version__)  # Version tested on sklearn.__version__ == 1.9.x
print(lgb.__version__)

### Utility Functions

In [ ]:
# Print each named variable as "name = value" with 4 significant digits
def pprint_var(**kwargs):
    for k, v in kwargs.items():
        print(f"{k} = {v:.4g}")

### 2.1 Data loading

One row per ICU stay. The data dictionary describes every column and assigns it to a category.

In [ ]:
DATA = Path("data/raw")
print(DATA)

In [ ]:
raw_df = pd.read_csv(DATA / "training_v2.csv")
dict_df = pd.read_csv(DATA / "WiDS Datathon 2020 Dictionary.csv")

print(raw_df.shape)
raw_df.head()

In [ ]:
raw_df.dtypes.value_counts()

In [ ]:
dict_df.query("`Variable Name` == 'd1_lactate_max'")

In [ ]:
category_of = dict_df.set_index("Variable Name")["Category"]
feature_groups = category_of.reindex(raw_df.columns).fillna(
    "not in dictionary"
)
feature_groups.value_counts()

*   `h1_*` = min/max during the **first hour**, `d1_*` = min/max during the **first day**, `*_apache` = the value used by the APACHE score (worst value of the first day)

*   Most features are numeric, 8 are text categories (sex, ethnicity, ...)

### 2.2 Target balance

In [ ]:
TARGET = "hospital_death"

death_rate = raw_df[TARGET].mean()
pprint_var(deaths=raw_df[TARGET].sum(), death_rate=death_rate)
pprint_var(accuracy_always_survives=1 - death_rate)

**Observations**

8.63% of the stays end in death (7,915 / 91,713). A model always answering "survives" reaches 91.37% accuracy while detecting no death at all, which is why the accuracy is not one of our metrics.

### 2.3 Missing values

In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.heatmap(
    raw_df.sample(2000, random_state=RANDOM_STATE).isna(),
    cbar=False,
    cmap="viridis",
    xticklabels=False,
    yticklabels=False,
)
ax.set(
    xlabel="186 columns",
    ylabel="2,000 random stays",
    title="Missing values (yellow)",
)
plt.show()

In [ ]:
missing_share = raw_df.isna().mean().sort_values(ascending=False)

print(
    f"Columns with missing values: {(missing_share > 0).sum()} / {raw_df.shape[1]}"
)
for limit in (0.5, 0.8, 0.9):
    print(
        f"  more than {limit:.0%} missing: {(missing_share > limit).sum()} columns"
    )
print(
    f"Rows with at least one missing value: {raw_df.isna().any(axis=1).mean():.2%}"
)

missing_share.head(8)

In [ ]:
n_missing_per_row = raw_df.isna().sum(axis=1)
missing_decile = pd.qcut(n_missing_per_row, 10, duplicates="drop")

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
missing_share.plot.hist(bins=30, ax=ax[0])
ax[0].set(
    xlabel="Share of missing values in the column",
    ylabel="Number of columns",
    title="Missingness per column",
)

missing_share.groupby(feature_groups).mean().sort_values().plot.barh(ax=ax[1])
ax[1].set(
    xlabel="Mean share missing",
    ylabel="",
    title="Missingness by feature category",
)

raw_df.groupby(missing_decile, observed=True)[TARGET].mean().plot.bar(
    ax=ax[2], color="tab:red"
)
ax[2].set(
    xlabel="Number of missing values in the row (deciles)",
    ylabel="Observed mortality",
    title="Mortality vs number of missing values",
)
ax[2].tick_params(axis="x", rotation=45)
plt.show()

**Observations**

We see that:

*   175 of the 186 columns have missing values, 74 are more than 50% missing. **99.97% of the rows** have at least one missing value, so dropping incomplete rows is impossible

*   **Missingness is informative** (right plot): patients with the *fewest* missing values have the *highest* mortality. Clinicians order more tests when they are worried, so the values are not missing at random

**Consequence:** we never use an imputation that hides that a value was missing. LightGBM and the random forest handle missing values natively, and the logistic regression gets a 0/1 "was missing" indicator.

### 2.4 Distributions and outliers

We compare a few clinically important variables between survivors and non-survivors.

In [ ]:
KEY_FEATURES = [
    "age",
    "d1_lactate_max",
    "gcs_motor_apache",
    "d1_sysbp_min",
    "d1_bun_max",
    "d1_spo2_min",
    "d1_heartrate_max",
    "bmi",
]

outcome_df = raw_df[KEY_FEATURES].assign(
    outcome=raw_df[TARGET].map({0: "survived", 1: "died"})
)

fig, ax = plt.subplots(2, 4, figsize=(18, 7))
for axis, feature in zip(ax.ravel(), KEY_FEATURES):
    sns.boxplot(
        data=outcome_df,
        x="outcome",
        y=feature,
        hue="outcome",
        showfliers=False,
        palette={"survived": "tab:blue", "died": "tab:red"},
        ax=axis,
    )
    axis.set(xlabel="", ylabel="", title=feature)
plt.tight_layout()
plt.show()

Are the extreme values errors? We look at the range of a few vital signs, and count a few impossible values.

In [ ]:
APACHE_PREDICTIONS = [
    "apache_4a_hospital_death_prob",
    "apache_4a_icu_death_prob",
]
VITALS = [
    "d1_heartrate_max",
    "d1_heartrate_min",
    "d1_temp_min",
    "d1_sysbp_min",
    "d1_resprate_max",
    "bmi",
]

display(raw_df[VITALS].describe().T)

# Pairs of columns (minimum, maximum) of the same measurement
MIN_MAX = [
    (col, col.replace("_min", "_max"))
    for col in raw_df.columns
    if col.endswith("_min")
]
n_inverted = sum(
    (raw_df[col_min] > raw_df[col_max]).sum() for col_min, col_max in MIN_MAX
)

pprint_var(
    share_aged_89=(raw_df.age == 89).mean(),
    negative_pre_icu_stays=(raw_df.pre_icu_los_days < 0).sum(),
    inverted_min_max=n_inverted,
    apache_equal_minus_1=(raw_df[APACHE_PREDICTIONS[0]] == -1).sum(),
)

**Observations**

*   As expected, the patients who died are older, have a higher lactate (poor tissue perfusion), a lower motor GCS (reduced consciousness), a lower minimum blood pressure and oxygen saturation, a higher urea (BUN, kidney function) and a higher heart rate. The BMI barely differs

*   The ranges of the vital signs are physiologically plausible for ICU patients (e.g. a minimum heart rate of 0 can be a cardiac arrest). Age is capped at 89, a usual de-identification rule

*   A few values are impossible and are fixed in Section 2.7: negative pre-ICU stays, minimum larger than maximum, and APACHE predictions equal to `-1`

### 2.5 Categorical variables and correlations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
for axis, column in zip(ax, ["apache_3j_bodysystem", "icu_admit_source"]):
    summary_df = (
        raw_df.groupby(column)[TARGET]
        .agg(["mean", "size"])
        .sort_values("mean")
    )
    summary_df.index = [
        f"{name} (n={size:,})" for name, size in summary_df["size"].items()
    ]
    summary_df["mean"].plot.barh(ax=axis, color="tab:red")
    axis.axvline(death_rate, color="k", ls="--", lw=0.8)
    axis.set(
        xlabel="Observed mortality (dashed = overall)", ylabel="", title=column
    )
plt.tight_layout()
plt.show()

**Is the task too easy?** For each numeric feature, we use its raw value directly as a score and compute the ROC-AUC (oriented to be ≥ 0.5). If one feature had an AUC close to 1, the task would be trivial.

In [ ]:
ID_COLUMNS = ["encounter_id", "patient_id", "hospital_id", "icu_id"]
numeric_columns = raw_df.select_dtypes("number").columns.drop(
    [TARGET] + ID_COLUMNS + APACHE_PREDICTIONS
)

univariate_auc = {}
for column in numeric_columns:
    known = raw_df[column].notna()
    auc = metrics.roc_auc_score(
        raw_df.loc[known, TARGET], raw_df.loc[known, column]
    )
    univariate_auc[column] = max(
        auc, 1 - auc
    )  # A low value can be as informative as a high one

univariate_auc = pd.Series(univariate_auc, name="ROC-AUC").sort_values(
    ascending=False
)
univariate_auc.head(10)

In [ ]:
corr_df = raw_df[numeric_columns].corr().abs()
n_pairs = (
    (corr_df > 0.95).sum().sum() - len(corr_df)
) // 2  # Remove the diagonal, count each pair once
pprint_var(pairs_with_corr_above_095=n_pairs)

In [ ]:
blood_pressure = [
    c
    for c in numeric_columns
    if c.startswith("d1_") and "bp" in c and "min" in c
]

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(raw_df[blood_pressure].corr(), dtype=bool))
ax = sns.heatmap(
    raw_df[blood_pressure].corr(),
    cbar=True,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    mask=mask,
)
ax.set(title="Correlation between first-day minimum blood pressures")
plt.show()

**Observations**

*   The best measured feature (`d1_lactate_max`) reaches a ROC-AUC of only **…** on its own. **No single feature gives the answer**, the task is not trivial

*   **…** pairs of features have $|r| > 0.95$ (e.g. `h1_*_min` and `h1_*_max` when a lab was measured only once, or invasive and non-invasive blood pressure). We keep them since trees are not harmed by redundancy and the logistic regression is regularized

### 2.6 Identifiers and leakage checks

Leakage = information that would not be available at prediction time or that makes the validation unrealistically easy.

In [ ]:
display(raw_df[ID_COLUMNS].nunique())

stays_per_hospital = raw_df.groupby("hospital_id").size()
mortality_per_hospital = raw_df.groupby("hospital_id")[TARGET].mean()
large = stays_per_hospital >= 100
pprint_var(
    median_stays_per_hospital=stays_per_hospital.median(),
    min_mortality_large_hospitals=mortality_per_hospital[large].min(),
    max_mortality_large_hospitals=mortality_per_hospital[large].max(),
)

# Only the hospital column of the unlabeled (Kaggle test) file is read
kaggle_test_hospitals = set(
    pd.read_csv(DATA / "unlabeled.csv", usecols=["hospital_id"]).hospital_id
)
shared = kaggle_test_hospitals & set(raw_df.hospital_id)
print(
    f"Kaggle test file: {len(kaggle_test_hospitals)} hospitals, only {len(shared)} also in training"
)

In [ ]:
apache_df = raw_df[APACHE_PREDICTIONS].where(raw_df[APACHE_PREDICTIONS] >= 0)
has_apache = apache_df.notna().all(axis=1)
pprint_var(share_with_apache=has_apache.mean())

for column in APACHE_PREDICTIONS:
    auc = metrics.roc_auc_score(
        raw_df.loc[has_apache, TARGET], apache_df.loc[has_apache, column]
    )
    pprint_var(**{f"auc_{column}": auc})

**Leakage decisions**

| Source | Finding | Decision |
|:--|:--|:--|
| `encounter_id`, `patient_id` | Unique per row (no patient appears twice) | Dropped |
| `hospital_id`, `icu_id` | 147 hospitals with very different mortality; the Kaggle test file contains mostly **other** hospitals | Not used as features, used as **groups** for splitting |
| `apache_4a_*_death_prob` | Output of an existing clinical model computed from the same data | Excluded from the features, kept as a **benchmark** |
| Timing | All features are collected during the first 24 h | Kept |

`readmission_status` is 0 for everybody, so it carries no information and is dropped.

### 2.7 Cleaning

*   Drop the constant column `readmission_status`

*   Merge "Undefined Diagnoses" and "Undefined diagnoses" (same category spelled twice)

*   APACHE predictions equal to `-1` → missing 

*   Negative `pre_icu_los_days` → missing (a negative duration is impossible)

*   Swap inverted min/max values (recording error)

*   Diagnosis codes → categories: 502.01 is a code, not a quantity. We keep the integer part (diagnosis group)

We don't remove more outliers. The provider already clipped extreme values and removing them would delete the sickest patients.

In [ ]:
icu_df = raw_df.drop(columns=["readmission_status"])
icu_df["apache_2_bodysystem"] = icu_df["apache_2_bodysystem"].replace(
    "Undefined Diagnoses", "Undefined diagnoses"
)

# -1 is a code for "APACHE could not be computed", not a probability
for col in APACHE_PREDICTIONS:
    icu_df.loc[icu_df[col] < 0, col] = np.nan

# A negative stay before ICU admission is impossible
icu_df.loc[icu_df["pre_icu_los_days"] < 0, "pre_icu_los_days"] = np.nan

# Swap the min and max values recorded in the wrong order
for col_min, col_max in MIN_MAX:
    inverted = icu_df[col_min] > icu_df[col_max]
    icu_df.loc[inverted, [col_min, col_max]] = icu_df.loc[
        inverted, [col_max, col_min]
    ].values

# Diagnosis codes are labels, not quantities: keep the diagnosis group as a text category
for col in ["apache_2_diagnosis", "apache_3j_diagnosis"]:
    icu_df[col] = (
        icu_df[col]
        .floordiv(1)
        .astype("Int64")
        .astype(str)
        .replace("<NA>", np.nan)
    )

print(icu_df.shape)

### 2.8 Features

*   Identifiers, target and APACHE predictions are removed

*   `n_missing_values` = number of missing measurements of the patient: a summary of *how intensively the patient was monitored* (Section 2.3)

*   The text categories are **one-hot encoded** with [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) (one 0/1 column per category). It does not use the target, so it can be done once, before the split

**Note:** missing values are kept as `NaN`. The random forest and LightGBM handle them natively; for the logistic regression they are imputed inside a pipeline (Section 3.3).

In [ ]:
features_df = icu_df.drop(columns=ID_COLUMNS + [TARGET] + APACHE_PREDICTIONS)
features_df["n_missing_values"] = features_df.isna().sum(axis=1)

CATEGORICAL = features_df.select_dtypes(exclude="number").columns.tolist()
xs_df = pd.get_dummies(features_df, columns=CATEGORICAL, dtype=float)

ys = icu_df[TARGET].values
groups = icu_df["hospital_id"].values  # Hospital of each stay

print(CATEGORICAL)
print(xs_df.shape, ys.shape, groups.shape)

### 2.9 Train / test split by hospital

We hold out 20% of the **hospitals** as a test set, never used for training or for choosing the models.

[`GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) works like `train_test_split`, but keeps all the stays of a hospital on the same side.

In [ ]:
splitter = model_selection.GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=RANDOM_STATE
)
train_idx, test_idx = next(splitter.split(xs_df, ys, groups))

train_xs, test_xs = xs_df.iloc[train_idx], xs_df.iloc[test_idx]
train_ys, test_ys = ys[train_idx], ys[test_idx]
train_groups, test_groups = groups[train_idx], groups[test_idx]

print(
    "train:",
    train_xs.shape,
    "-",
    len(set(train_groups)),
    "hospitals, mortality",
    f"{train_ys.mean():.4f}",
)
print(
    "test: ",
    test_xs.shape,
    "-",
    len(set(test_groups)),
    "hospitals, mortality",
    f"{test_ys.mean():.4f}",
)

---

## 3. Training

### 3.1 Validation strategy

All the model choices are made with a **5-fold cross-validation grouped by hospital** ([`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)) on the training set: the validation hospitals are never in the training folds, as for a new hospital.

We use [`cross_validate`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) with `return_train_score=True`, so that we get for each fold:

*   The ROC-AUC on the **training folds** (patients the model has seen)

*   The ROC-AUC on the **validation fold** (new hospitals)

The difference between the two measures the **overfitting**.

In [ ]:
cv = model_selection.GroupKFold(n_splits=5)
SCORING = "roc_auc"

cv_results = {}  # Model name -> scores of each fold


def evaluate(name, model, xs=train_xs):
    scores = model_selection.cross_validate(
        model,
        xs,
        train_ys,
        groups=train_groups,
        cv=cv,
        scoring=SCORING,
        return_train_score=True,
    )
    cv_results[name] = scores
    print(
        f"{name}: train {scores['train_score'].mean():.4f} | validation {scores['test_score'].mean():.4f} ± {scores['test_score'].std():.4f}"
    )

### 3.2 Baseline

A [`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) predicts the training death rate for everybody: ROC-AUC = 0.5 by construction. Any useful model must beat it.

In [ ]:
# 2. Instantiation
model = dummy.DummyClassifier(strategy="prior")

# 3. Learn (fit) the model and 4. predict, on each fold
evaluate("Dummy", model)

### 3.3 Logistic Regression

[`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) is linear, fast and interpretable. It needs a [`pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html):

*   `imputer`: missing values replaced by the median, **plus a 0/1 "was missing" column** for each feature (`add_indicator=True`), so that the model can still use the missingness

*   `scaler`: standardization, so that the regularization treats all the features equally

We choose the regularization strength `C` with a [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
lr_pipe = pipeline.Pipeline(
    steps=[
        (
            "imputer",
            impute.SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", preprocessing.StandardScaler()),
        ("model", linear_model.LogisticRegression(max_iter=2000)),
    ]
)

param_grid = {"model__C": [0.001, 0.01, 0.1, 1.0]}

search = model_selection.GridSearchCV(
    lr_pipe, param_grid, scoring=SCORING, cv=cv, n_jobs=-1, verbose=1
)
search.fit(train_xs, train_ys, groups=train_groups)

print(f"Best parameter (CV score: {search.best_score_:.4f})")
print(search.best_params_)

In [ ]:
plt.bar(
    [str(c) for c in param_grid["model__C"]],
    search.cv_results_["mean_test_score"],
    yerr=search.cv_results_["std_test_score"],
)
plt.ylim(0.85, 0.91)
plt.xlabel("C")
plt.ylabel("ROC-AUC (new hospitals)")
plt.grid(axis="y")
plt.show()

In [ ]:
lr_model = search.best_estimator_
evaluate("Logistic regression", lr_model)

**Observations**

*   The values of `C` give close scores: with more than 70,000 training stays, the regularization matters little (except when it is too strong)

*   The logistic regression reaches **…** on new hospitals: already a strong baseline. The gap between train and validation is small: a linear model cannot memorize the patients

### 3.4 Random Forest

A [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) averages many decision trees. It handles missing values natively and does not need scaling.

We tune `min_samples_leaf`, the minimum number of patients in a leaf: the smaller it is, the more the trees can memorize the training patients. With `return_train_score=True`, the grid search also gives the training score.

In [ ]:
# 2. Instantiation
model = ensemble.RandomForestClassifier(
    n_estimators=200, max_samples=0.5, n_jobs=-1, random_state=RANDOM_STATE
)

param_grid = {"min_samples_leaf": [1, 10, 50]}

search = model_selection.GridSearchCV(
    model,
    param_grid,
    scoring=SCORING,
    cv=cv,
    return_train_score=True,
    verbose=1,
)
search.fit(train_xs, train_ys, groups=train_groups)

print(f"Best parameter (CV score: {search.best_score_:.4f})")
print(search.best_params_)

In [ ]:
labels = [str(v) for v in param_grid["min_samples_leaf"]]
x = np.arange(len(labels))

plt.bar(
    x - 0.2, search.cv_results_["mean_train_score"], width=0.4, label="train"
)
plt.bar(
    x + 0.2,
    search.cv_results_["mean_test_score"],
    width=0.4,
    yerr=search.cv_results_["std_test_score"],
    label="validation (new hospitals)",
)
plt.xticks(x, labels)
plt.ylim(0.8, 1.01)
plt.xlabel("min_samples_leaf")
plt.ylabel("ROC-AUC")
plt.legend()
plt.show()

In [ ]:
rf_model = search.best_estimator_
evaluate("Random forest", rf_model)

### 3.5 LightGBM

[LightGBM](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html) is a gradient boosting library: trees are added one at a time, each one correcting the errors of the previous ones. It is the model family used by all the top Kaggle teams, and it handles missing values natively.

We tune `num_leaves`, the size of each tree (4 leaves = a very simple tree of 2–3 questions).

In [ ]:
# 2. Instantiation
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    colsample_bytree=0.5,  # Each tree sees half of the features
    subsample=0.8,  # Each tree sees 80% of the patients
    subsample_freq=1,
    random_state=RANDOM_STATE,
    verbose=-1,
)

param_grid = {"num_leaves": [4, 15, 31, 63]}

search = model_selection.GridSearchCV(
    model,
    param_grid,
    scoring=SCORING,
    cv=cv,
    return_train_score=True,
    verbose=1,
)
search.fit(train_xs, train_ys, groups=train_groups)

print(f"Best parameter (CV score: {search.best_score_:.4f})")
print(search.best_params_)

In [ ]:
labels = [str(v) for v in param_grid["num_leaves"]]
x = np.arange(len(labels))

plt.bar(
    x - 0.2, search.cv_results_["mean_train_score"], width=0.4, label="train"
)
plt.bar(
    x + 0.2,
    search.cv_results_["mean_test_score"],
    width=0.4,
    yerr=search.cv_results_["std_test_score"],
    label="validation (new hospitals)",
)
plt.xticks(x, labels)
plt.ylim(0.8, 1.01)
plt.xlabel("num_leaves")
plt.ylabel("ROC-AUC")
plt.legend()
plt.show()

In [ ]:
lgbm_model = search.best_estimator_
evaluate("LightGBM", lgbm_model)

### 3.6 Comparison

In [ ]:
MODELS = ["Dummy", "Logistic regression", "Random forest", "LightGBM"]

summary_df = pd.DataFrame(
    {
        "train": [cv_results[name]["train_score"].mean() for name in MODELS],
        "validation": [
            cv_results[name]["test_score"].mean() for name in MODELS
        ],
        "validation std": [
            cv_results[name]["test_score"].std() for name in MODELS
        ],
    },
    index=MODELS,
)
summary_df["train - validation"] = (
    summary_df["train"] - summary_df["validation"]
)
summary_df.round(4)

In [ ]:
plt.boxplot([cv_results[name]["test_score"] for name in MODELS[1:]])
plt.xticks([1, 2, 3], MODELS[1:])
plt.ylabel("ROC-AUC on each validation fold")
plt.title("Fold-to-fold variability (new hospitals)")
plt.show()

**Observations**

*   **LightGBM is the best model** (**…** on new hospitals), followed by the logistic regression (**…**) and the random forest (**…**)

*   The **fold-to-fold standard deviation (≈ …)** is about the size of the spread of the whole Kaggle top 50 (0.005): differences in the third decimal are mostly noise

### 3.7 Overfitting source 1 — the training patients

The `train - validation` column of the table above shows how much each model **memorizes its training patients**.

**Observations**

*   The logistic regression has a small gap (**…**): a linear model cannot memorize individual patients

*   The trees have a much larger gap: random forest **…**, LightGBM **…**. As seen in the grid searches, the smaller the leaves (random forest) or the bigger the trees (LightGBM), the higher the training score, without improving the validation score

*   This is the classic overfitting: the more flexible the model, the more it learns the noise of the training patients. It is still the best model on new hospitals, but its training score must never be used to judge it

### 3.8 Overfitting source 2 — the training hospitals

We compare two ways of splitting the training set:

*   **New hospitals**: `GroupKFold`, as everywhere else

*   **Same hospitals**: [`KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) with shuffling: the validation patients come from hospitals seen in training

We also add `hospital_id` as a feature. With a random split, the model can learn the death rate of each hospital; with a grouped split, the validation hospitals are unknown, so this cannot help.

In [ ]:
random_cv = model_selection.KFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)
train_hosp_xs = train_xs.assign(
    hospital_id=pd.Categorical(train_groups)
)  # A category, so LightGBM can learn one value per hospital

rows = {}
for split_name, split_cv in [
    ("new hospitals", cv),
    ("same hospitals", random_cv),
]:
    for feature_name, xs in [
        ("without hospital_id", train_xs),
        ("with hospital_id", train_hosp_xs),
    ]:
        scores = model_selection.cross_val_score(
            lgbm_model,
            xs,
            train_ys,
            groups=train_groups,
            cv=split_cv,
            scoring=SCORING,
        )
        rows[(split_name, feature_name)] = scores.mean()

pd.Series(rows, name="ROC-AUC").unstack().round(4)

**Observations**

*   Without `hospital_id`, the score on new patients of the same hospitals (**…**) is only slightly higher than on new hospitals (**…**): on average, the patient signal generalizes well between hospitals

*   With `hospital_id`, the score **…** with the random split but **…** with the grouped split: the model learns the death rate of each training hospital, which looks useful when the validation patients come from the same hospitals, but **does not transfer to a new hospital**. This is overfitting the training hospitals

*   A random split would have pushed us to keep a feature that hurts in a new hospital. This is why all our choices are made with `GroupKFold`, and why `hospital_id` is not a feature

### 3.9 Overfitting source 3 — model complexity

Gradient boosting adds trees one at a time. With [`validation_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html), we follow the training and validation scores as the number of trees grows.

In [ ]:
N_TREES = [50, 100, 200, 500, 1000, 2000]

train_scores, valid_scores = model_selection.validation_curve(
    lgbm_model,
    train_xs,
    train_ys,
    groups=train_groups,
    param_name="n_estimators",
    param_range=N_TREES,
    cv=model_selection.GroupKFold(n_splits=3),
    scoring=SCORING,
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(N_TREES, train_scores.mean(axis=1), "o--", label="train")
plt.plot(
    N_TREES,
    valid_scores.mean(axis=1),
    "o-",
    label="validation (new hospitals)",
)
plt.xscale("log")
plt.xlabel("Number of trees")
plt.ylabel("ROC-AUC")
plt.title("Training vs validation score")
plt.legend()
plt.show()

pd.DataFrame(
    {
        "train": train_scores.mean(axis=1),
        "validation": valid_scores.mean(axis=1),
    },
    index=N_TREES,
).round(4)

**Observations**

*   The training score keeps increasing towards 1: with enough trees, the model **memorizes** the training patients

*   The validation score increases, reaches a maximum (**…** trees), then stays flat or slowly decreases: adding trees after this point only adds overfitting

*   Our model uses 500 trees, close to the best point

### 3.10 Overfitting source 4 — hospital "fingerprints" in the features

Removing `hospital_id` does not remove the hospital: each hospital has its own case mix, units and testing habits. If the features reveal *which hospital* a patient comes from, the model can learn hospital-specific shortcuts.

*   **Test 1:** can a classifier recognize the hospital (among the 20 largest) from the features? From the missing-value pattern only (was this measurement taken: yes/no)?

*   **Test 2:** how well does the missing-value pattern alone predict death?

In [ ]:
largest = pd.Series(train_groups).value_counts().head(20).index
in_largest = np.isin(train_groups, largest)

hosp_xs = train_xs[in_largest]
hosp_ys = train_groups[in_largest]
missing_xs = hosp_xs.isna().astype(int)  # 1 = the measurement is missing

model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, verbose=-1)
for name, xs in [
    ("all features", hosp_xs),
    ("missing-value pattern only", missing_xs),
]:
    scores = model_selection.cross_val_score(
        model, xs, hosp_ys, cv=3, scoring="accuracy"
    )
    print(f"Hospital recognized from {name}: {scores.mean():.1%}")

pprint_var(
    accuracy_largest_hospital=pd.Series(hosp_ys)
    .value_counts(normalize=True)
    .max()
)

In [ ]:
evaluate(
    "LightGBM on missingness only", lgbm_model, train_xs.isna().astype(int)
)

**Observations**

*   **The features reveal the hospital**: among 20 hospitals, the hospital is recognized **…** of the time (vs **…** by always guessing the largest one), and **…** from the missing-value pattern alone

*   **The missing-value pattern alone predicts death** with a ROC-AUC of **…**, better than the best single measured value (Section 2.5). It encodes the clinicians' judgement (sicker patients get more tests), but also local practices

*   **Consequence:** the model partly relies on *how doctors work* in each hospital. This is why it must be validated on new hospitals

### 3.11 Overfitting source 5 — preprocessing fitted on all the data

Many Kaggle solutions replace a category by the **death rate of that category** ("target encoding", [`TargetEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)). It uses the target, so it must be learnt **on the training folds only**.

We show what happens otherwise with a very fine category: *diagnosis × ICU* (thousands of small groups).

*   **Naive:** the encoder is fitted once on the whole training set, then the usual cross-validation is run

*   **Correct:** the encoder is inside a pipeline, so it is re-fitted on the training folds of each split

In [ ]:
diag_icu = (
    icu_df["apache_3j_diagnosis"].astype(str)
    + "_"
    + icu_df["icu_id"].astype(str)
).values[train_idx]
pprint_var(groups=len(set(diag_icu)))

# Naive: fit on all the training set (the validation patients' own outcome is used)
encoder = preprocessing.TargetEncoder(random_state=RANDOM_STATE)
encoder.fit(diag_icu.reshape(-1, 1), train_ys)
naive_xs = train_xs.assign(
    diag_icu_death_rate=encoder.transform(diag_icu.reshape(-1, 1)).ravel()
)

# Correct: the encoder is a step of the pipeline
correct_pipe = pipeline.Pipeline(
    steps=[
        (
            "encoder",
            compose.make_column_transformer(
                (
                    preprocessing.TargetEncoder(random_state=RANDOM_STATE),
                    ["diag_icu"],
                ),
                remainder="passthrough",
            ),
        ),
        ("model", lgbm_model),
    ]
)

evaluate("LightGBM + naive encoding", lgbm_model, naive_xs)
evaluate(
    "LightGBM + correct encoding",
    correct_pipe,
    train_xs.assign(diag_icu=diag_icu),
)

**Observations**

*   **The naive encoding gives …, far above the Kaggle winner (0.915)!** This score is fake: each validation patient's own outcome is included in the death rate of its group, so the new feature partly *contains the answer*

*   Done correctly, the same feature gives **…**, no better than without it (**…**): with tiny groups, the encoding is mostly noise

*   **Lesson:** every step that uses the target must be inside the cross-validation (in a pipeline). This is the same rule as for the PCA in the course: fit on the training set only

### 3.12 APACHE predictions as features

Same LightGBM, with the two APACHE IVa probabilities added as features.

In [ ]:
train_apache_xs = train_xs.join(icu_df[APACHE_PREDICTIONS])
evaluate("LightGBM + APACHE", lgbm_model, train_apache_xs)

**Observations**

Adding the APACHE IVa predictions does **not** improve the model (**…** vs **…**): APACHE is a fixed formula of first-day variables that are already in our features, so it brings no new information. Our model does not use them, so it stays independent of the score we compare to (Section 4.5).

### 3.13 Does our preprocessing help?

Our teacher suggested to approach the state of the art with **simple models and a careful preprocessing**, rather than with large ensembles. We check two preprocessing choices:

*   The **"was missing" indicators** of the logistic regression (`add_indicator=True` vs `False`)

*   The **`n_missing_values`** feature for LightGBM (with vs without)

In [ ]:
lr_no_indicator = base.clone(lr_model).set_params(imputer__add_indicator=False)
evaluate("Logistic regression, no missing indicators", lr_no_indicator)

evaluate(
    "LightGBM, no n_missing_values",
    lgbm_model,
    train_xs.drop(columns="n_missing_values"),
)

In [ ]:
reference = pd.Series(
    {
        "Kaggle 1st place (stacking)": 0.91497,
        "Kaggle 50th place": 0.91013,
        "Kaggle 14th, best single LightGBM": 0.9097,
        "Ren et al. 2022 (stacking)": 0.879,
    }
)
ours = pd.Series(
    {
        name: cv_results[name]["test_score"].mean()
        for name in ["Logistic regression", "Random forest", "LightGBM"]
    }
)
comparison = pd.concat([ours, reference]).sort_values()

plt.figure(figsize=(8, 5))
colors = [
    "tab:gray" if name in reference.index else "tab:blue"
    for name in comparison.index
]
plt.barh(comparison.index, comparison.values, color=colors)
plt.xlim(0.85, 0.92)
plt.xlabel("ROC-AUC")
plt.title("Our models (blue, new hospitals) vs reported results (gray)")
plt.show()

**Observations**

*   The missing indicators help the logistic regression (**…** vs **…**): the information "this test was not ordered" is useful

*   `n_missing_values` **…** LightGBM (**…** vs **…**)

*   **Comparison with the state of the art:** our single LightGBM (**…**) is about **…** below the best single LightGBM reported on Kaggle (0.9097, with 700+ features and a random split) and the winner's 50-model stack (0.915, on a different test set). This gap is about the size of our fold-to-fold standard deviation

**Selected model for Section 4:** LightGBM (best score on new hospitals), compared with the logistic regression (simplest model).

---

## 4. Evaluation on unseen hospitals

The models are now re-trained on the **whole training set** and evaluated **once** on the test hospitals.

### 4.1 Test scores

In [ ]:
final_models = {
    "Logistic regression": lr_model,
    "Random forest": rf_model,
    "LightGBM": lgbm_model,
}

test_prob_ys = {}
rows = {}
for name, model in final_models.items():
    # 3. Learn (fit) the model
    model.fit(train_xs, train_ys)
    # 4. Predict on new data
    test_prob_ys[name] = model.predict_proba(test_xs)[:, 1]
    rows[name] = {
        "train ROC-AUC": metrics.roc_auc_score(
            train_ys, model.predict_proba(train_xs)[:, 1]
        ),
        "CV ROC-AUC": cv_results[name]["test_score"].mean(),
        "test ROC-AUC": metrics.roc_auc_score(test_ys, test_prob_ys[name]),
        "test PR-AUC": metrics.average_precision_score(
            test_ys, test_prob_ys[name]
        ),
        "test Brier": metrics.brier_score_loss(test_ys, test_prob_ys[name]),
    }

pd.DataFrame(rows).T.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, model in final_models.items():
    metrics.RocCurveDisplay.from_estimator(
        model, test_xs, test_ys, name=name, ax=ax[0]
    )
    metrics.PrecisionRecallDisplay.from_estimator(
        model, test_xs, test_ys, name=name, ax=ax[1]
    )
ax[0].set(title="ROC curve (test hospitals)")
ax[1].set(title="Precision-recall curve (test hospitals)")
plt.show()

**Observations**

*   LightGBM is the best model on the test hospitals too: ROC-AUC **…**, PR-AUC **…** (chance level = death rate, **…**)

*   All the models lose some performance between the training set, the cross-validation and the test set. The trees have the largest gap between train and test: they overfit the most, but still generalize best

*   The test score is slightly **…** than the CV estimate: choosing the best settings on the same validation folds makes the CV score a bit optimistic, and the test hospitals are a different sample

### 4.2 Decision threshold

**Clinical goal (our assumption):** the model is a *screening* tool that flags patients for a closer review. Missing a patient who will die is worse than reviewing a patient who survives, so we want to **catch at least 80% of the deaths** (recall ≥ 0.80).

The threshold is chosen on the **training set**, with out-of-fold predictions ([`cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)): each training patient is predicted by a model that did not see their hospital.

In [ ]:
TARGET_RECALL = 0.80

oof_prob_ys = model_selection.cross_val_predict(
    lgbm_model,
    train_xs,
    train_ys,
    groups=train_groups,
    cv=cv,
    method="predict_proba",
)[:, 1]

precision, recall, thresholds = metrics.precision_recall_curve(
    train_ys, oof_prob_ys
)
threshold = thresholds[
    recall[:-1] >= TARGET_RECALL
].max()  # Highest threshold with recall >= 80%
pprint_var(threshold=threshold)

In [ ]:
pred_ys = test_prob_ys["LightGBM"] >= threshold

print(
    metrics.classification_report(
        test_ys, pred_ys, target_names=["survived", "died"]
    )
)
pprint_var(alert_rate=pred_ys.mean())

In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_predictions(
    test_ys, pred_ys, display_labels=["survived", "died"], cmap=plt.cm.Blues
)
disp.figure_.suptitle("Confusion Matrix (LightGBM, test hospitals)")
plt.grid(False)
plt.show()

**Observations**

*   The threshold chosen on the training hospitals **transfers to the new hospitals**: the recall of the deaths is **…** (target 0.80)

*   The price is an alert for **…** of the patients, with a precision of **…**: about **1 alert in …** is a patient who will die, vs 1 in 11.5 if we flagged patients at random

*   The accuracy (**…**) is lower than the "always survives" baseline (91%): this is expected, since we accept false alarms to catch more deaths. It confirms that the accuracy is not the right metric here

### 4.3 Calibration

A calibration curve groups the patients by predicted risk and plots the observed death rate in each group ([`CalibrationDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html)). Points on the diagonal mean that the probabilities can be read literally ("30% risk" = 30% of such patients die).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name in final_models:
    calibration.CalibrationDisplay.from_predictions(
        test_ys,
        test_prob_ys[name],
        n_bins=10,
        strategy="quantile",
        name=name,
        ax=ax,
    )
ax.set(title="Calibration (test hospitals)")
plt.show()

**Observations**

**…** (are the curves close to the diagonal? which model is the closest?)

### 4.4 Most important features

*   For LightGBM: `feature_importances_` with `importance_type="gain"` = how much each feature improved the trees in total

*   For the logistic regression: the coefficients $a_j$ of the standardized features, as in the course (positive = higher risk)

In [ ]:
lgbm_importance = pd.Series(
    lgbm_model.booster_.feature_importance(importance_type="gain"),
    index=train_xs.columns,
)
lgbm_importance = lgbm_importance / lgbm_importance.sum()

lr_coefs = pd.Series(
    lr_model.named_steps["model"].coef_[0],
    index=lr_model.named_steps["imputer"].get_feature_names_out(),
)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
lgbm_importance.nlargest(15).sort_values().plot.barh(ax=ax[0])
ax[0].set(title="LightGBM: share of total gain", xlabel="")
lr_coefs.sort_values(key=abs).tail(15).plot.barh(ax=ax[1])
ax[1].set(title="Logistic regression: coefficients (standardized)", xlabel="")
plt.tight_layout()
plt.show()

**Observations**

**…** (the top features should be clinically sensible: age, ventilation, Glasgow scores, lactate, urea, oxygen saturation, diagnosis...)

**Note:** as in the course, correlated features share their importance, and an important feature is not necessarily a *cause* of death.

### 4.5 Comparison with APACHE IVa

APACHE IVa is the score already used in hospitals. We compare it with our model on the test patients that have an APACHE score: discrimination, number of alerts needed to catch 80% of the deaths, and calibration.

In [ ]:
test_apache_ys = icu_df[APACHE_PREDICTIONS[0]].values[test_idx]
has_score = ~np.isnan(test_apache_ys)

train_apache_ys = icu_df[APACHE_PREDICTIONS[0]].values[train_idx]
train_has_score = ~np.isnan(train_apache_ys)
_, apache_recall, apache_thresholds = metrics.precision_recall_curve(
    train_ys[train_has_score], train_apache_ys[train_has_score]
)
apache_threshold = apache_thresholds[apache_recall[:-1] >= TARGET_RECALL].max()

for name, prob_ys, thr in [
    ("APACHE IVa", test_apache_ys[has_score], apache_threshold),
    ("LightGBM", test_prob_ys["LightGBM"][has_score], threshold),
]:
    alerts = prob_ys >= thr
    print(
        f"{name:12s} ROC-AUC {metrics.roc_auc_score(test_ys[has_score], prob_ys):.4f} | recall {metrics.recall_score(test_ys[has_score], alerts):.3f} | alert rate {alerts.mean():.3f}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
calibration.CalibrationDisplay.from_predictions(
    test_ys[has_score],
    test_apache_ys[has_score],
    n_bins=10,
    strategy="quantile",
    name="APACHE IVa",
    ax=ax,
)
calibration.CalibrationDisplay.from_predictions(
    test_ys[has_score],
    test_prob_ys["LightGBM"][has_score],
    n_bins=10,
    strategy="quantile",
    name="LightGBM",
    ax=ax,
)
ax.set(title="Calibration: APACHE IVa vs LightGBM")
plt.show()

**Observations**

*   **Discrimination:** LightGBM **…** vs APACHE IVa **…** on the same patients

*   **Same clinical goal, fewer alerts:** to catch 80% of the deaths, APACHE must alert **…** of the patients, our model **…**

*   **Calibration:** **…** (APACHE IVa is expected to over-estimate the risk: a fixed formula developed on another population and period)

### 4.6 Errors and hospitals

*   **Missed deaths:** how do the deaths that the model missed (false negatives) differ from the detected ones?

*   **Per hospital:** is the model as good in every test hospital?

In [ ]:
deaths = test_ys == 1
error_df = test_xs.loc[
    deaths,
    [
        "age",
        "ventilated_apache",
        "gcs_motor_apache",
        "d1_lactate_max",
        "n_missing_values",
        "elective_surgery",
    ],
]
error_df = error_df.assign(
    outcome=np.where(pred_ys[deaths], "detected death", "missed death")
)
error_df.groupby("outcome").mean().T.round(2)

In [ ]:
hospital_rows = []
for hospital in set(test_groups):
    mask = test_groups == hospital
    if test_ys[mask].sum() >= 10:  # Enough deaths to compute a ROC-AUC
        hospital_rows.append(
            {
                "stays": mask.sum(),
                "ROC-AUC": metrics.roc_auc_score(
                    test_ys[mask], test_prob_ys["LightGBM"][mask]
                ),
                "observed mortality": test_ys[mask].mean(),
                "mean predicted risk": test_prob_ys["LightGBM"][mask].mean(),
            }
        )
hospital_df = pd.DataFrame(hospital_rows)
display(hospital_df.describe().round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(hospital_df["stays"], hospital_df["ROC-AUC"])
ax[0].set(
    xlabel="Stays in the hospital",
    ylabel="ROC-AUC",
    title="Discrimination per hospital",
)
ax[1].scatter(
    hospital_df["mean predicted risk"],
    hospital_df["observed mortality"],
    color="tab:red",
)
ax[1].plot([0, 0.2], [0, 0.2], "k--", lw=0.8)
ax[1].set(
    xlabel="Mean predicted risk",
    ylabel="Observed mortality",
    title="Calibration per hospital",
)
plt.show()

**Observations**

*   **Missed deaths look like survivors on day 1**: **…** (less often ventilated, better Glasgow score, lower lactate, more elective surgery, more missing values). They were probably stable during the first 24 hours and deteriorated later: a limit of first-day data

*   **Per hospital**, the ROC-AUC goes from **…** to **…**, and some hospitals are mis-calibrated (right plot): **this is where the hospital effect shows**. A local check, and possibly a recalibration, would be needed before using the model in a new hospital

---

## 5. Conclusion

### What we did

We predicted in-hospital death from the first 24 hours of 91,713 ICU stays. Following our teacher's feedback, we did not try to beat the leaderboard but asked **where the models overfit** and **how far simple models can go**. All the choices were made with a cross-validation grouped by hospital, and the final evaluation was done once on hospitals never seen in training, with APACHE IVa as a clinical benchmark.

### What worked

*   **LightGBM** is the best model: ROC-AUC **…** on new hospitals, about **…** below the best single model reported on Kaggle, with one model and no feature engineering

*   It **beats APACHE IVa** on the same patients (**…** vs **…**) and needs fewer alerts to catch 80% of the deaths

*   The **logistic regression** with missing indicators is a strong, simple baseline (**…**)

### Sources of overfitting we identified

| Source | Evidence | What we did |
|:--|:--|:--|
| Training patients | Trees: train **…** vs validation **…** | Judge models only on new hospitals; tune leaf/tree size |
| Training hospitals | `hospital_id` helps with a random split, not with a grouped split | `GroupKFold`, test set of unseen hospitals, no hospital identifiers |
| Model complexity | More trees: train → 1, validation flat | Number of trees close to the best validation point |
| Hospital fingerprints | Hospital recognized **…** of the time from the features | Grouped validation, local check before use |
| Preprocessing on all the data | Naive target encoding: fake **…** | Every step that uses the target inside a pipeline |

### What did not work

*   The random forest never beat the logistic regression; the APACHE predictions as features brought nothing

*   About 20% of the deaths are missed by design (our threshold), mostly patients who looked stable on day 1

*   The performance varies between hospitals

### If we had more time

1.  **Leave-one-hospital-out** evaluation, and a recalibration of the model for each new hospital

2.  **Clinical feature engineering** (total Glasgow score, shock index, first-hour vs first-day changes), as in the top Kaggle solutions

3.  **External validation** on another dataset (e.g. MIMIC or eICU on PhysioNet)

### References

*   WiDS Datathon 2020 data: [Kaggle](https://www.kaggle.com/competitions/widsdatathon2020/data); [PhysioNet](https://physionet.org/content/widsdatathon2020/1.0.0/) (Lee, Raffa, Ghassemi et al.)

*   Kaggle leaderboard and write-ups: links in Section 1.3

*   Ren, N., Zhao, X., Zhang, X. (2022). Mortality prediction in ICU using a stacked ensemble model. *Computational and Mathematical Methods in Medicine*

*   Caires Silveira, E., et al. (2022). Prediction of hospital mortality in intensive care unit patients from clinical and laboratory data: a machine learning approach. *World Journal of Critical Care Medicine* 11(5)

*   Lundberg, S. M., Lee, S.-I. (2017). A unified approach to interpreting model predictions (SHAP). *NeurIPS*

*   Vickers, A. J., Elkin, E. B. (2006). Decision curve analysis. *Medical Decision Making* 26(6)

---